In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
from tensorflow.keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

c:\Users\Playdata\multimodal\multi_venv\Lib\site-packages\tf_keras\src\datasets\cifar.py:32: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [ ]:
# 프루닝 적용
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.sparsity import keras as sparsity

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation="relu", input_shape=(32,32,3)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])

# 가중치 크기가 작은 것부터 0으로 만들어 희소화 해주는 프루닝
pruned_model = sparsity.prune_low_magnitude(
    model,  # 프루닝 적용할 모델
    pruning_schedule=sparsity.PolynomialDecay(  # 학습 스텝에 따라 sparsity를 점진적으로 증가
        initial_sparsity=0.0,  # 시작 sparsity는 0%
        final_sparsity=0.5,    # 마지막 sparsity는 50% (가중치를 절반은 0으로)
        begin_step=0,  # 프루닝 시작 스텝
        end_step=1000  # 프루닝 종료 스텝 (이후에는 그냥 유지)
    )
)

# 프루닝 스텝 업데이트용 콜백
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

pruned_model.fit(x_train/255.0, y_train, epochs=2, callbacks=callbacks)




Epoch 1/2


1563/1563 [==============================] - 14s 8ms/step - loss: 1.4883 - accuracy: 0.4716
Epoch 2/2
1563/1563 [==============================] - 12s 7ms/step - loss: 1.2227 - accuracy: 0.5720


In [4]:
# 양자화 (QAT)
quantize_model = tfmot.quantization.keras.quantize_model

quantized_model = quantize_model(model)

quantized_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

quantized_model.fit(x_train/255.0, y_train, epochs=2)

Epoch 1/2
1563/1563 [==============================] - 16s 10ms/step - loss: 1.1747 - accuracy: 0.5885
Epoch 2/2
1563/1563 [==============================] - 15s 10ms/step - loss: 1.0841 - accuracy: 0.6214


In [6]:
# 모델 평가
pruned_loss, pruned_acc = pruned_model.evaluate(x_test / 255.0, y_test)
print('프루닝 모델 정확도', pruned_acc)
print('프루닝 모델 손실', pruned_loss)

quantized_loss, quantized_acc = quantized_model.evaluate(x_test / 255.0, y_test)
print('양자화 모델 정확도', quantized_acc)
print('양자화 모델 손실', quantized_loss)

313/313 [==============================] - 1s 3ms/step - loss: 1.2023 - accuracy: 0.5783
프루닝 모델 정확도 0.5782999992370605
프루닝 모델 손실 1.2023108005523682
313/313 [==============================] - 1s 3ms/step - loss: 1.1055 - accuracy: 0.6174
양자화 모델 정확도 0.6173999905586243
양자화 모델 손실 1.1054582595825195


In [ ]:
# 모델 크기 비교

# 기본 모델 저장
model.save('model.h5')

# 프루닝된 모델 저장
pruned_model_stripped = tfmot.sparsity.keras.strip_pruning(pruned_model)
pruned_model_stripped.save('pruned_model.h5')

# 양자화된 모델 저장
converter = tf.lite.TFLiteConverter.from_keras_model(quantized_model)
quantized_tflite_model = converter.convert()

with open('quantized_model.tflite', 'wb') as f:
    f.write(quantized_tflite_model)

import os

print('기본 모델 크기:', os.path.getsize('model.h5')/1e6, 'MB')
print('프루닝된 모델 크기:', os.path.getsize('pruned_model.h5')/1e6, 'MB')
print('양자화된 모델 크기:', os.path.getsize('quantized_model.tflite')/1e6, 'MB')

c:\Users\Playdata\multimodal\multi_venv\Lib\site-packages\tf_keras\src\engine\training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


INFO:tensorflow:Assets written to: C:\Users\Playdata\AppData\Local\Temp\tmpa7qj4mt8\assets


INFO:tensorflow:Assets written to: C:\Users\Playdata\AppData\Local\Temp\tmpa7qj4mt8\assets


기본 모델 크기: 1.86904 MB
프루닝된 모델 크기: 1.86904 MB
양자화된 모델 크기: 1.85564 MB
